## Introduction
 the popularity of more powerful libraries such as PyToch and TensorFlow, they are not easy to use and have a steep learning curve. So, for people who are just starting to learn deep learning, there is no better library to use other than the Keras library.

Keras is a high-level API for building deep learning models. It has gained favor for its ease of use and syntactic simplicity facilitating fast development.  


## Objectives
* How to use the Keras library to build a regression model
* Download and clean the data set
* Build a neural network
* Train and test the network     



In [ ]:
# All Libraries required for this lab are listed below.

!pip install numpy==2.0.2
!pip install pandas==2.2.2
!pip install tensorflow_cpu==2.18.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.3/230.3 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 81.7 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml_dtypes 0.5.3
    Uninstalling ml_dtypes-0.5.3:
      Successfully uninstalled ml_dtypes-0.5.3
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling tensorboard-2.19.0:
      Successfully uninstalled tensorboard-2.19.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorstore 0.1.76 requires ml_dtypes>=0.5.0, but you have ml-dtypes 0.4.1 which is incompatible.
tensorflow 2.19.0 requires ml-dtypes<1.0.0,>=0.5.1, but you have ml-dtypes 0.4.1 which is incompatible.
tensorflow 2.19.0 requires tensorboard~=2.19.0, bu

#### To use Keras, you will also need to install a backend framework – such as TensorFlow.

If you install TensorFlow 2.16 or above, it will install Keras by default.

We are using the CPU version of tensorflow since we are dealing with smaller datasets.
You may install the GPU version of tensorflow on your machine to accelarate the processing of larger datasets


#### Suppress the tensorflow warning messages
We use the following code to  suppress the warning messages due to use of CPU architechture for tensoflow.



In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
import pandas as pd
import numpy as np
import keras

import warnings
warnings.simplefilter('ignore', FutureWarning)

<strong>The dataset is about the compressive strength of different samples of concrete based on the volumes of the different ingredients that were used to make them. Ingredients include:</strong>

* Cement
* Blast furnace slag
* Fly ash
* Water
* Superplasticizer
* Coarse aggregate
* Fine aggregate

## Download and Clean the Data Set


In [ ]:
filepath='https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0101EN/labs/data/concrete_data.csv'
concrete_data = pd.read_csv(filepath)

concrete_data.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


So the first concrete sample has 540 cubic meter of cement, 0 cubic meter of blast furnace slag, 0 cubic meter of fly ash, 162 cubic meter of water, 2.5 cubic meter of superplaticizer, 1040 cubic meter of coarse aggregate, 676 cubic meter of fine aggregate. Such a concrete mix which is 28 days old, has a compressive strength of 79.99 MPa.


#### Let's check how many data points we have


In [ ]:
concrete_data.shape

(1030, 9)

So, there are approximately 1000 samples to train our model on. Because of the few samples, we have to be careful not to overfit the training data.


In [ ]:
#Let's check the dataset for any missing values.
concrete_data.describe()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136,35.817961
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912,16.705742
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.710000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000,34.445000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000,46.135000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


In [ ]:
concrete_data.isnull().sum()

,0
Cement,0
Blast Furnace Slag,0
Fly Ash,0
Water,0
Superplasticizer,0
Coarse Aggregate,0
Fine Aggregate,0
Age,0
Strength,0


The data looks very clean and is ready to be used to build our model.

#### Split data into predictors and target
The target variable in this problem is the concrete sample strength. Therefore, our predictors will be all the other columns.


In [ ]:
concrete_data_columns = concrete_data.columns

In [ ]:
predictors = concrete_data[concrete_data_columns[concrete_data_columns != 'Strength']] # all columns except Strength
target = concrete_data['Strength'] # Strength column

Let's do a quick sanity check of the predictors and the target dataframes.


In [ ]:
predictors.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360


In [ ]:
target.head()

,Strength
0,79.99
1,61.89
2,40.27
3,41.05
4,44.30


the last step is to normalize the data by substracting the mean and dividing by the standard deviation.

In [ ]:
predictors_norm = (predictors - predictors.mean()) / predictors.std()
predictors_norm.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,2.476712,-0.856472,-0.846733,-0.916319,-0.620147,0.862735,-1.217079,-0.279597
1,2.476712,-0.856472,-0.846733,-0.916319,-0.620147,1.055651,-1.217079,-0.279597
2,0.491187,0.795140,-0.846733,2.174405,-1.038638,-0.526262,-2.239829,3.551340
3,0.491187,0.795140,-0.846733,2.174405,-1.038638,-0.526262,-2.239829,5.055221
4,-0.790075,0.678079,-0.846733,0.488555,-1.038638,0.070492,0.647569,4.976069


Let's save the number of predictors to n_cols since we will need this number when building our network.

In [ ]:
n_cols = predictors_norm.shape[1] # number of predictors

##  Import Keras Packages

##### Let's import the rest of the packages from the Keras library that we will need to build our regression model.


In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Input

## Build a Neural Network


a function that defines our regression model for us so that we can conveniently call it to create our model.

In [ ]:
# define regression model
def regression_model():
    # create model
    model = Sequential()
    model.add(Input(shape=(n_cols,)))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1))

    # compile model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

The above function create a model that has two hidden layers, each of 50 hidden units.

##Train and Test the Network

In [ ]:
# build the model
model = regression_model()

Next, we will train and test the model at the same time using the fit method. We will leave out 30% of the data for validation and we will train the model for 100 epochs.

In [ ]:
# fit the model
model.fit(predictors_norm, target, validation_split=0.3, epochs=100, verbose=2)

Epoch 1/100
23/23 - 2s - 76ms/step - loss: 1661.1423 - val_loss: 1166.2311
Epoch 2/100
23/23 - 0s - 18ms/step - loss: 1509.8994 - val_loss: 1050.0156
Epoch 3/100
23/23 - 0s - 14ms/step - loss: 1280.8727 - val_loss: 877.4674
Epoch 4/100
23/23 - 0s - 6ms/step - loss: 956.1530 - val_loss: 657.5986
Epoch 5/100
23/23 - 0s - 8ms/step - loss: 610.0024 - val_loss: 435.6628
Epoch 6/100
23/23 - 0s - 12ms/step - loss: 359.3704 - val_loss: 284.0742
Epoch 7/100
23/23 - 0s - 7ms/step - loss: 267.1884 - val_loss: 217.0011
Epoch 8/100
23/23 - 0s - 15ms/step - loss: 238.2930 - val_loss: 193.9499
Epoch 9/100
23/23 - 0s - 11ms/step - loss: 222.1221 - val_loss: 181.5575
Epoch 10/100
23/23 - 0s - 13ms/step - loss: 209.4364 - val_loss: 177.5397
Epoch 11/100
23/23 - 0s - 12ms/step - loss: 199.9795 - val_loss: 173.5096
Epoch 12/100
23/23 - 0s - 11ms/step - loss: 192.1285 - val_loss: 168.1465
Epoch 13/100
23/23 - 0s - 10ms/step - loss: 185.9230 - val_loss: 164.5346
Epoch 14/100
23/23 - 0s - 17ms/step - loss: 1

*Now using the same dateset,try to recreate regression model featuring five hidden layers, each with 50 nodes and ReLU activation functions, a single output layer, optimized using the Adam optimizer.*

In [ ]:
def regression_model():
    input_colm = predictors_norm.shape[1] # Number of input features
    # create model
    model = Sequential()
    model.add(Input(shape=(input_colm,)))  # Set the number of input features
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1))  # Output layer

    # compile model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

*Train and evaluate the model simultaneously using the fit() method by reserving 10% of the data for validation and training the model for 100 epochs*

In [ ]:
# build the model
model = regression_model()
model.fit(predictors_norm, target, validation_split=0.1, epochs=100, verbose=2)


Epoch 1/100
29/29 - 2s - 82ms/step - loss: 1512.1696 - val_loss: 936.2356
Epoch 2/100
29/29 - 0s - 14ms/step - loss: 625.3254 - val_loss: 325.3275
Epoch 3/100
29/29 - 0s - 11ms/step - loss: 243.1626 - val_loss: 218.6601
Epoch 4/100
29/29 - 0s - 10ms/step - loss: 203.8357 - val_loss: 189.1303
Epoch 5/100
29/29 - 0s - 5ms/step - loss: 180.5589 - val_loss: 175.0406
Epoch 6/100
29/29 - 0s - 5ms/step - loss: 163.2351 - val_loss: 160.7077
Epoch 7/100
29/29 - 0s - 6ms/step - loss: 148.0456 - val_loss: 146.1484
Epoch 8/100
29/29 - 0s - 10ms/step - loss: 132.9905 - val_loss: 134.2872
Epoch 9/100
29/29 - 0s - 10ms/step - loss: 120.1224 - val_loss: 114.4414
Epoch 10/100
29/29 - 0s - 5ms/step - loss: 105.4518 - val_loss: 117.0876
Epoch 11/100
29/29 - 0s - 5ms/step - loss: 89.8941 - val_loss: 123.7556
Epoch 12/100
29/29 - 0s - 5ms/step - loss: 78.1472 - val_loss: 92.9071
Epoch 13/100
29/29 - 0s - 5ms/step - loss: 68.4039 - val_loss: 83.7644
Epoch 14/100
29/29 - 0s - 14ms/step - loss: 58.0876 - val_